# Determine and Download Datasets

first we load our data from the csv into a pandas dataframe

In [1]:
import os
print(os.getcwd())

/Users/jonaselsper/Documents/Programming/github/master-thesis/data-science-agent


In [2]:
import pandas as pd

df = pd.read_csv("../notebooks/catalog_distributions.csv")
df.head()

,title,rdf_about,distributions
0,Dynamisches Angebot der Taxi Dortmund eG,https://mobilithek.info//offers/63691097729758...,[{'uri': 'https://ckan.govdata.de/dataset/d7ff...
1,dynamische Detektordaten Stadt Rüsselsheim,https://mobilithek.info//offers/11000000000318...,[{'uri': 'https://ckan.govdata.de/dataset/7996...
2,dwista - Tafeln dynamische Wegweiser mit inte...,https://mobilithek.info//offers/-2019997857864...,[{'uri': 'https://ckan.govdata.de/dataset/e26c...
3,dwista - Tafeln dynamische Wegweiser mit inte...,https://mobilithek.info//offers/-7644162386486...,[{'uri': 'https://ckan.govdata.de/dataset/7d08...
4,Durchfahrtsbeschränkungen für Dieselfahrzeuge ...,https://mobilithek.info//offers/-1713828761512...,[{'uri': 'https://ckan.govdata.de/dataset/6a3a...


In [3]:
import ast

print(list(df.distributions)[0])

[{'uri': 'https://ckan.govdata.de/dataset/d7ffe65c-c473-46a9-8a03-98bf3fba57e4/resource/af104411-a1b1-4eb4-a96c-c6632e6fbcca', 'title': 'Dynamisches Angebot der Taxi Dortmund eG', 'access_url': 'https://mobilithek.info//offers/636910977297584128#accessURL', 'download_url': None, 'format': None, 'media_type': None}]


in one run, pandas recognized the distributions column as a string, so we need to convert it back to a list

In [4]:
df['distributions'] = df['distributions'].apply(lambda x: ast.literal_eval(x) if pd.notnull(x) else [])

we now iterate over all distributions and count the media types to get a feeling what types exist and which of them are tabular

In [5]:
from collections import Counter

media_type_counter = Counter()


def tabular(x):
    if not isinstance(x, list):
        return

    for item in x:
        if not isinstance(item, dict):
            continue

        if item.get("media_type"):
            media_type_counter[item["media_type"]] += 1


df['distributions'].apply(tabular)

print("Media-Type Distribution:")
for media_type, count in media_type_counter.most_common():
    print(f"{count}: {media_type}")

Media-Type Distribution:
135915: application/x-netcdf
52411: https://www.iana.org/assignments/media-types/text/csv
34167: text/csv
26312: application/pdf
25612: application/zip
14814: https://www.iana.org/assignments/media-types/text/xml
11902: https://www.iana.org/assignments/media-types/text/xlsx
10274: text/plain
7826: application/vnd.openxmlformats-officedocument.spreadsheetml.sheet
6490: application/xml
6220: https://www.iana.org/assignments/media-types/application/pdf
4947: https://www.iana.org/assignments/media-types/application/json
4120: https://www.iana.org/assignments/media-types/application/zip
2839: https://www.iana.org/assignments/media-types/application/geo+json
2163: application/json
2140: text/html
2125: application/vnd.ms-excel
1927: text/csv+extended
1649: https://www.iana.org/assignments/media-types/application/vnd.openxmlformats-officedocument.spreadsheetml.sheet
958: image/tiff
925: https://www.iana.org/assignments/media-types/application/gml+xml
905: https://www.

we pick the tabular media types by hand and save them as constants in a list

In [6]:
tabular_media_types = [
    "https://www.iana.org/assignments/media-types/text/csv",
    "text/csv",
    "https://www.iana.org/assignments/media-types/text/xlsx",
    "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet",
    "application/vnd.ms-excel",
    "text/csv+extended",
    "https://www.iana.org/assignments/media-types/application/vnd.openxmlformats-officedocument.spreadsheetml.sheet",
    "https://www.iana.org/assignments/media-types/application/vnd.ms-excel",
    "https://www.iana.org/assignments/media-types/application/vnd.oasis.opendocument.spreadsheet",
    "application/vnd.oasis.opendocument.spreadsheet",
    "text/tab-separated-values",
    "application/xlsx",
    "https://www.iana.org/assignments/media-types/application/json/application/vnd.openxmlformats-officedocument.spreadsheetml.sheet"
]

In [7]:
from collections import Counter

tabular_media_set = set(tabular_media_types)

total_counter = Counter()
tabular_counter = Counter()

def analyze_distributions(x):
    if not isinstance(x, list):
        return

    for item in x:
        if not isinstance(item, dict):
            continue

        media_type = item.get("media_type")
        if media_type:
            total_counter[media_type] += 1

            # Nur zählen wenn in tabular-Liste
            if media_type in tabular_media_set:
                tabular_counter[media_type] += 1


df['distributions'].apply(analyze_distributions)

total_all = sum(total_counter.values())
total_tabular = sum(tabular_counter.values())

print(f"Tabular Media Types ({total_tabular} von {total_all} -> {total_tabular/total_all*100:.1f}%):")
for mt, count in tabular_counter.most_common():
    pct = count / total_all * 100
    print(f"  {count} ({pct:.1f}%): {mt}")

Tabular Media Types (112474 von 364192 -> 30.9%):
  52411 (14.4%): https://www.iana.org/assignments/media-types/text/csv
  34167 (9.4%): text/csv
  11902 (3.3%): https://www.iana.org/assignments/media-types/text/xlsx
  7826 (2.1%): application/vnd.openxmlformats-officedocument.spreadsheetml.sheet
  2125 (0.6%): application/vnd.ms-excel
  1927 (0.5%): text/csv+extended
  1649 (0.5%): https://www.iana.org/assignments/media-types/application/vnd.openxmlformats-officedocument.spreadsheetml.sheet
  407 (0.1%): https://www.iana.org/assignments/media-types/application/vnd.ms-excel
  35 (0.0%): https://www.iana.org/assignments/media-types/application/vnd.oasis.opendocument.spreadsheet
  17 (0.0%): application/vnd.oasis.opendocument.spreadsheet
  5 (0.0%): text/tab-separated-values
  2 (0.0%): application/xlsx
  1 (0.0%): https://www.iana.org/assignments/media-types/application/json/application/vnd.openxmlformats-officedocument.spreadsheetml.sheet


fot the study / experiment, we start with csv datasets

In [8]:
csv_media_types = [
    "https://www.iana.org/assignments/media-types/text/csv",
    "text/csv",
    "text/csv+extended"
]

In [9]:
def has_tabular(distributions):
    for dist in distributions:
        if not isinstance(dist, dict):
            continue
        media_type = dist.get("media_type")
        if media_type in tabular_media_types:
            return True
    return False


df['has_tabular'] = df['distributions'].apply(has_tabular)
df["has_tabular"]

0         False
1         False
2         False
3         False
4         False
          ...  
147254    False
147255    False
147256    False
147257    False
147258    False
Name: has_tabular, Length: 147259, dtype: bool

In [10]:
def has_csv(distributions):
    for dist in distributions:
        if not isinstance(dist, dict):
            continue
        media_type = dist.get("media_type")
        if media_type in csv_media_types:
            return True
    return False

df['has_csv'] = df['distributions'].apply(has_csv)
df["has_csv"]

0         False
1         False
2         False
3         False
4         False
          ...  
147254    False
147255    False
147256    False
147257    False
147258    False
Name: has_csv, Length: 147259, dtype: bool

query all datasets that contain csv distributions

In [11]:
df.query("has_csv == True")

,title,rdf_about,distributions,has_tabular,has_csv
85,Detektierte Erschütterungen an der LKW-Auflieg...,https://mobilithek.info//offers/57321669459012...,[{'uri': 'https://ckan.govdata.de/dataset/fd05...,True,True
127,Dauerzählstellen (Rad) Hamburg,https://mobilithek.info//offers/-3869360234918...,[{'uri': 'https://ckan.govdata.de/dataset/19a4...,True,True
128,Dauerzählstellen (Rad) Hamburg,https://mobilithek.info//offers/-8196963435227...,[{'uri': 'https://ckan.govdata.de/dataset/0fe5...,True,True
132,Datengrundlage zur Stoppzeitanalyse auf der le...,https://mobilithek.info//offers/93810129412002...,[{'uri': 'https://ckan.govdata.de/dataset/25f2...,True,True
134,Daten zur Evaluation des Leezenflow-Systems in...,https://opendata.stadt-muenster.de/dataset/dat...,[{'uri': 'https://opendata.stadt-muenster.de/d...,True,True
...,...,...,...,...,...
145163,Ansprechpartnerinnen zur Vergabe einer BNR-ZD ...,https://service.brandenburg.de/service/de/adre...,[{'uri': 'https://service.brandenburg.de/servi...,True,True
145164,Forschungseinrichtungen im Agrarbereich,https://service.brandenburg.de/service/de/adre...,[{'uri': 'https://service.brandenburg.de/servi...,True,True
145217,Ansprechstellen des Zolls für die Bekämpfung v...,https://ckan.govdata.de/dataset/308a7cde-23e9-...,[{'uri': 'https://ckan.govdata.de/dataset/308a...,True,True
145218,Ansprechstellen des Zolls für die Festsetzung ...,https://ckan.govdata.de/dataset/ef42d9fa-5710-...,[{'uri': 'https://ckan.govdata.de/dataset/ef42...,True,True


from those datasets we now pick 50 randomly for the study

In [12]:
df_csv_samples = df.query("has_csv == True").sample(50, random_state=42)

df_csv_samples

,title,rdf_about,distributions,has_tabular,has_csv
108398,KFÜ-Messwerte,https://opendata.schleswig-holstein.de/dataset...,[{'uri': 'https://opendata.schleswig-holstein....,True,True
116051,Betriebe im Verarbeitenden Gewerbe nach Wirtsc...,https://region.statistik-nord.de/detail_timeli...,[{'uri': 'https://region.statistik-nord.de/det...,True,True
142955,Messergebnisse zur Radioaktivität in: Sojabohn...,http://suche.transparenz.hamburg.de/dataset/db...,[{'uri': 'http://suche.transparenz.hamburg.de/...,True,True
139606,Messergebnisse zur Radioaktivität in: Schweine...,http://suche.transparenz.hamburg.de/dataset/56...,[{'uri': 'http://suche.transparenz.hamburg.de/...,True,True
118595,Geborenen- (+) bzw. Gestorbenenüberschuss (-) ...,https://region.statistik-nord.de/detail_timeli...,[{'uri': 'https://region.statistik-nord.de/det...,True,True
12930,"München, Landeshauptstadt: Bevölkerung",https://open.bydata.de/api/hub/repo/datasets/1...,[{'uri': 'https://open.bydata.de/api/hub/repo/...,True,True
135982,Messergebnisse zur Radioaktivität in: Reinwass...,http://suche.transparenz.hamburg.de/dataset/72...,[{'uri': 'http://suche.transparenz.hamburg.de/...,True,True
16317,Messdaten Gamma-ODL: Station Kornbach,https://open.bydata.de/api/hub/repo/datasets/o...,[{'uri': 'https://open.bydata.de/api/hub/repo/...,True,True
16265,Messdaten Gamma-ODL: Station Dorfprozelten,https://open.bydata.de/api/hub/repo/datasets/o...,[{'uri': 'https://open.bydata.de/api/hub/repo/...,True,True
13979,Sünching: Baufertigstellungen,https://open.bydata.de/api/hub/repo/datasets/3...,[{'uri': 'https://open.bydata.de/api/hub/repo/...,True,True


now we have to randomly choose one csv distribution per dataset to download. this is only needed if there are multiple csv distributions per dataset

In [13]:
import numpy as np

np.random.seed(42)

def select_random_csv(distributions):
    if not isinstance(distributions, list):
        return None

    csv_distributions = [
        item for item in distributions
        if isinstance(item, dict) and item.get("media_type") and "csv" in item["media_type"].lower()
    ]

    if not csv_distributions:
        return None

    return csv_distributions[np.random.randint(len(csv_distributions))]

df_csv_samples["study_distribution"] = df_csv_samples["distributions"].apply(select_random_csv)

In [14]:
df_csv_samples["study_distribution"]

108398    {'uri': 'https://opendata.schleswig-holstein.d...
116051    {'uri': 'https://region.statistik-nord.de/deta...
142955    {'uri': 'http://suche.transparenz.hamburg.de/d...
139606    {'uri': 'http://suche.transparenz.hamburg.de/d...
118595    {'uri': 'https://region.statistik-nord.de/deta...
12930     {'uri': 'https://open.bydata.de/api/hub/repo/d...
135982    {'uri': 'http://suche.transparenz.hamburg.de/d...
16317     {'uri': 'https://open.bydata.de/api/hub/repo/d...
16265     {'uri': 'https://open.bydata.de/api/hub/repo/d...
13979     {'uri': 'https://open.bydata.de/api/hub/repo/d...
15539     {'uri': 'https://open.bydata.de/api/hub/repo/d...
110165    {'uri': 'https://opendata.schleswig-holstein.d...
119718    {'uri': 'https://region.statistik-nord.de/deta...
62290     {'uri': 'https://region.statistik-nord.de/deta...
12586     {'uri': 'https://open.bydata.de/api/hub/repo/d...
104336    {'uri': 'https://opendata.schleswig-holstein.d...
66622     {'uri': 'https://www.landesdat

In [15]:
df_csv_samples.to_csv("../playground/notebooks/csv_study_datasets.csv", index=True)

OSError: Cannot save file into a non-existent directory: '../playground/notebooks'

let's extract the download urls for the selected distributions

In [ ]:
df_csv_samples["download_url"] = df_csv_samples["study_distribution"].apply(
    lambda x: x.get("access_url") if isinstance(x, dict) else None
)

In [ ]:
df_csv_samples["download_url"]

next step is to generate the dataset title, like it's done in CKAN, to create the rdf metadata url for downloading the metadata rdf file

In [ ]:
import re

def munge_title_to_name(title, max_length=91):
    """
    Munge a title into a name suitable for use in a URL.
    Based on CKAN's munge_title_to_name function.
    """
    if not title:
        return ""

    title = str(title)

    # Convert to lowercase
    title = title.lower()

    # Replace German umlauts and special characters
    title = title.replace('﷿﷿', 'a')
    title = title.replace('﷿﷿', 'o')
    title = title.replace('﷿﷿', 'u')
    title = title.replace('﷿﷿', 'ss')
    title = title.replace('﷿﷿', 'a')
    title = title.replace('﷿﷿', 'o')
    title = title.replace('﷿﷿', 'u')

    # Replace non-alphanumeric characters (except hyphens, underscores) with spaces
    title = re.sub(r'[^\w\s-]', '', title)

    # Replace spaces and underscores with hyphens
    title = re.sub(r'[\s_]+', '-', title)

    # Collapse multiple hyphens
    title = re.sub('-+', '-', title)

    title = title.rstrip('-')

    # Limit length
    if len(title) > max_length:
        title = title[:max_length]


    return title

# Apply to dataframe
df_csv_samples["url_slug"] = df_csv_samples["title"].apply(
    lambda x: munge_title_to_name(x, max_length=95)
)

In [ ]:
df_csv_samples["url_slug"]

In [ ]:
base_url = "https://ckan.govdata.de/dataset/"

df_csv_samples["rdf_url"] = (
    base_url +
    df_csv_samples["url_slug"] +
    ".rdf"
)

df_csv_samples["rdf_url"]

we got the dataset download url and the metadata url.

now we create for each dataset a folder and download the csv file into that folder and its corresponding metadata rdf file

In [ ]:
import os
import requests
from pathlib import Path

output_dir = Path("../playground/study")
output_dir.mkdir(exist_ok=True)

session = requests.Session()

def download_file(session, url, filepath, label):
    try:
        response = session.get(url, timeout=90)
        response.raise_for_status()
        with open(filepath, "wb") as f:
            f.write(response.content)
        print(f"{label} ﷿﷿﷿")
        return True
    except Exception as e:
        print(f"{label} ﷿﷿﷿ - {e}")
        return False

for idx, row in df_csv_samples.iterrows():
    row_dir = Path(os.path.join(output_dir, str(idx)))
    row_dir.mkdir(exist_ok=True)

    # Download CSV
    download_file(
        session,
        row.get("download_url"),
        os.path.join(row_dir, "dataset.csv"),
        f"Row {idx}: CSV"
    )

    # Download RDF
    download_file(
        session,
        row.get("rdf_url"),
        os.path.join(row_dir, "metadata.rdf"),
        f"Row {idx}: RDF"
    )

session.close()

we tried to retrie all data, for the datasets with rdf we try to handle it by hand, its probably a url generation error

In [ ]:
df_csv_samples